# ResNet generalists (RQ1 baseline)

Trains **one ResNet per leave-one-out fold** on all seen generators together: the single-detector baseline
for the ResNet pools, like `meta/generalist.py` is for CLIP. See `resnet/generalist.py`.

- **Folds:** the 7 single folds plus `SD+wukong` (for `merge_siblings=True`), so 8 networks.
- **Train:** the expert recipe (`resnet/train.py`) on the `expert_train` images of every seen expert together,
  `expert_val` of the seen experts for early stopping and the temperature. Held-out generators are never seen.
- **Score:** every image of `meta_val` and `test` with every generalist → feature set `resnet18_generalists`.
- **Evaluate:** same columns as the meta-learners (`meta.loo`): threshold from seen `meta_val`, AUROC etc. on the
  held-out generators of `EVAL_POOL`.

**Time:** each generalist trains on 6x the images of one expert (~60k per epoch), so expect roughly 6x the
expert time per epoch. ResNet-18 is ~2.5x faster than ResNet-50 (~17 min per epoch on an M4 for ResNet-50).

**Safe to stop and rerun:** finished generalists and scored images are saved and skipped.
Set `LIMIT = 100` first for a test run of the whole notebook; it saves to separate folders.


In [1]:
import sys
sys.path.insert(0, "../..")  # repo root, so `common`, `data`, `meta` and `resnet` are importable

import matplotlib.pyplot as plt
import pandas as pd

from data import config
from resnet.generalist import FOLDS, run_generalist, score, train_all

ARCH = "resnet50"  # ~2.5x faster than resnet50, and the expert pools barely differed
LIMIT = None  # e.g. 100 for a test run: 100 images per label, pool and expert, saved separately
EPOCHS = 1 if LIMIT else 8
BATCH_SIZE = 64  # training; lower to 32 if you run out of memory
NUM_WORKERS = 4
TRAIN_FOLDS = None  # None = all of FOLDS, or e.g. ["ADM", "BigGAN"]
MERGE_SIBLINGS = True  # evaluate with SD and wukong held out together
EVAL_POOL = "test"  # "meta_val" while developing, so test stays untouched

print("dataset:", config.dataset_dir(), config.dataset_dir().exists())
print("folds:", FOLDS)


dataset: /Users/tim/Developer/SC4001-Metalearners-Route-or-Pool/input/genimage_d8889a9c True
folds: {'ADM': ['ADM'], 'BigGAN': ['BigGAN'], 'glide': ['glide'], 'Midjourney': ['Midjourney'], 'SD': ['SD'], 'VQDM': ['VQDM'], 'wukong': ['wukong'], 'SD+wukong': ['SD', 'wukong']}


## 1 · Train and calibrate one generalist per fold

In [ ]:
infos = train_all(TRAIN_FOLDS, epochs=EPOCHS, limit=LIMIT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, arch=ARCH)

[14:32:50] [ADM] resnet50 on mps: 60000 train images (50% fake), 12000 val images, generators ['BigGAN', 'glide', 'Midjourney', 'stable_diffusion_v_1_4', 'stable_diffusion_v_1_5', 'VQDM', 'wukong']
[14:33:53]   epoch 1 batch 50/937: train loss 0.6873, 51 images/s


### Training curves

Validation loss is on the `expert_val` images of the fold's seen experts. `T` is the calibration temperature.

In [ ]:
# Categorical slots in fixed order, one per fold.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#7a7a7a"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for color, (fold, info) in zip(COLORS, infos.items()):
    history = pd.DataFrame(info["history"])
    axes[0].plot(history.epoch, history.train_loss, color=color, lw=2, marker="o", ms=4, label=fold)
    axes[1].plot(history.epoch, history.val_loss, color=color, lw=2, marker="o", ms=4, label=fold)
for ax, title in zip(axes, ["Training loss", "Validation loss (seen expert_val)"]):
    ax.set_title(title, loc="left")
    ax.set_xlabel("epoch")
    ax.set_ylabel("BCE loss")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(frameon=False, fontsize=8, ncol=2, title="held out")
plt.tight_layout()
plt.show()

pd.DataFrame([{
    "held out": fold, "n train": info["n_train"], "best epoch": info["best_epoch"], "epochs run": info["epochs_run"],
    "val auc": info["val"]["auc"], "val acc": info["val"]["acc"], "val loss": info["val"]["loss"],
    "calibrated loss": info["val_calibrated_loss"], "T": info["temperature"],
    "minutes": info["train_seconds"] / 60,
} for fold, info in infos.items()]).round(4)


## 2 · Score `meta_val` and `test` with every generalist

Needs all 8 generalists trained.

In [ ]:
score(limit=LIMIT, num_workers=NUM_WORKERS, arch=ARCH)

## 3 · Leave-one-out results

One row per fold: `auc`, `ap`, `acc` on the held-out generators of `EVAL_POOL`, `seen_auc` on the others,
`gap` = `seen_auc` − `auc`. Same columns as the meta-learners and the CLIP generalist.

In [ ]:
run_generalist(MERGE_SIBLINGS, EVAL_POOL, limit=LIMIT, arch=ARCH).round(4)